# Modelling AusRAP: a road-safety dataset made of codes

> **Attribution.** AusRAP data © State of Victoria (Department of Transport and
> Planning), licensed [CC-BY 4.0](https://creativecommons.org/licenses/by/4.0/).
> Source: <https://opendata.transport.vic.gov.au/dataset/australian-road-assessment-program-ausrap>

The second hands-on notebook on schemas. [Notebook 5](5-schema-models.ipynb) taught
the pieces (fields, types, enums, rules) on a toy `RoadSafetyRating`. Here you use
them on a real dataset: **AusRAP Victoria**, the star ratings that Victoria's
Department of Transport and Planning publishes for its road network, produced with
the [iRAP](https://irap.org/) methodology.

The notebook ends in a schema: a Pydantic model for one AusRAP record, with every
coded column as a documented enum, checked against every row of the real file.
It stops there. Turning a schema into an Overture extension, and matching AusRAP to
Overture's roads, are separate jobs.

Sections:

1. Get the data and see what is in it.
2. Codes or labels? Look before designing.
3. What a record means: 100 m bins, worst-within-100 m.
4. Enums: one per coded column, each value documented.
5. The record: structs, a rule, and a geometry.
6. Check the schema against the data.
7. Emit the schema.

Some cells are *meant* to fail. A failure prints a `ValidationError`; the last lines
name the field and the rule it broke.

In [ ]:
%xmode Minimal
import urllib.request
from pathlib import Path

import duckdb

# Resolve paths from the notebook's location, so it works whether Jupyter was
# launched from the repo root or from notebooks/.
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
DATA = REPO_ROOT / "data"
RESULTS = REPO_ROOT / "results"

con = duckdb.connect()

## 1. Get the data

The full state file is about 80 MB, so it is not in the repo. The cell below
downloads it once into `data/` (which is gitignored) and reuses it afterwards.

In [ ]:
AUSRAP_URL = (
    "https://opendata.transport.vic.gov.au/dataset/"
    "da201a82-372b-4f23-8e88-9819a6672bd0/resource/"
    "4148542e-6e7b-476f-bcb2-de94114bc40a/download/ausrap.csv"
)
AUSRAP = DATA / "ausrap.csv"

if not AUSRAP.exists():
    DATA.mkdir(exist_ok=True)
    print("Downloading AusRAP Victoria (~80 MB, CC-BY 4.0, one time)...")
    urllib.request.urlretrieve(AUSRAP_URL, AUSRAP)

con.execute(
    f"CREATE TABLE ausrap AS SELECT * FROM read_csv_auto('{AUSRAP.as_posix()}', sample_size=-1)"
)
con.sql("SELECT count(*) AS rows, count(DISTINCT \"Location ID\") AS location_ids FROM ausrap")

63 columns, one row per 100 m of one carriageway. `Location ID` is unique per row.
First, which columns hold what kind of value?

In [ ]:
con.sql("""
    SELECT column_type, count(*) AS columns, list(column_name ORDER BY column_name)[:4] AS first_four
    FROM (DESCRIBE ausrap) GROUP BY column_type ORDER BY columns DESC
""")

## 2. Codes or labels?

Most columns are `BIGINT`: the coded ones describe road features (median type, speed
limit, roadside hazard), yet hold integers. Count the values in two of them.

In [ ]:
con.sql('SELECT "Median type" AS median_type, count(*) AS rows FROM ausrap GROUP BY 1 ORDER BY 1')

In [ ]:
con.sql('SELECT "Speed limit" AS speed_limit, count(*) AS rows FROM ausrap GROUP BY 1 ORDER BY 1')

These are **iRAP numeric codes**, not labels and not measurements. The speed limit
column holds `1, 3, 5, 7, ...`, not kilometres per hour. A value means nothing until
you look it up, and the lookup table is the part a schema has to carry.

**Where the meanings come from.** The [iRAP Coding
Manual](https://irap.org/specifications/) defines each attribute's codes. This
notebook uses Version 5.3, Drive on Left edition (2022), section 3 for the
definitions and section 4 (Quick Coding Guide) for the tables. Two caveats on that
choice:

- The Austroads copy of the *Star Rating and Investment Plan Manual* v3.10 sits
  behind a bot check and could not be fetched, so the codes here come from the
  Coding Manual alone. The two manuals describe the same coding scheme, but this
  notebook did not compare them.
- The dataset does not state which edition it was coded under (the survey dates
  are 2022 and 2023), and the manual's changelog shows code sets have changed:
  new pedestrian crossing categories and school zone attributes in 2013, and
  corrections for differential speeds and refuge islands in 2020. Section 6 checks
  that every code in the file is defined by the 5.3 tables, which is evidence the
  editions agree here, not proof.

**Which meanings were checked against the data.** Reading a manual proves what the
manual says. Two meanings can be tested against AusRAP itself:

1. Speed limit codes. Rows whose `Landmark` is a speed-limit sign ("Speed Limit 60
   km/hr") give a posted limit to compare with the code. For each code, the most
   common sign:

In [ ]:
con.sql("""
    WITH signs AS (
        SELECT "Speed limit" AS code,
               regexp_extract(Landmark, 'Speed Limit (\\d+)', 1)::INT AS sign_kmh,
               count(*) AS rows
        FROM ausrap WHERE Landmark LIKE 'Speed Limit%'
        GROUP BY 1, 2
    )
    SELECT code, sign_kmh, rows FROM signs
    QUALIFY rows = max(rows) OVER (PARTITION BY code)
    ORDER BY code
""")

Every code with more than a handful of signed rows lines up with the manual's
table: 3 is 40 km/h, 5 is 50, 7 is 60, 9 is 70, 11 is 80, 13 is 90, 15 is 100, 17
is 110. Code 1 ("under 30") has three signed rows. Signs and codes disagree on
some rows, as expected: a code summarises a whole 100 m, a landmark is one point.

2. Roadworks. The manual says "star ratings will not be generated" for major road
works (code 3). The data agrees:

In [ ]:
con.sql("""
    SELECT Roadworks AS roadworks_code,
           count(*) AS rows,
           count(*) FILTER (WHERE "Vehicle Star Rating Smoothed" IS NULL) AS without_star_rating
    FROM ausrap GROUP BY 1 ORDER BY 1
""")

Every other value's meaning rests on the manual alone. That includes the
`Vehicle SRS ...` score columns: the Coding Manual does not define scores (the Star
Rating manual does), so their descriptions in the schema below come from the column
names and are marked as such.

## 3. What a record means

Three facts about iRAP data decide the schema's shape:

- **One row is a 100 m bin.** Every row has `Length` 0.1 (km), a start coordinate
  (`Latitude`, `Longitude`) and an end coordinate (`End Latitude`, `End Longitude`).
  The geometry is the straight line between them: a `LineString`.
- **Each attribute is the worst case within the 100 m.** The manual: "if a 100 m
  segment includes a cliff for 50 m and a safety barrier for the next 50 m, record
  the cliff." A value describes a bin, not a place. Its enum values are listed
  from highest to lowest risk for this reason.
- **Each carriageway is its own row set.** A divided road is surveyed once per
  direction (`Carriageway` 1 and 2); an undivided road once (code 3). Sides are
  relative to the direction of survey: "driver-side" and "passenger-side".

There is no persistent segment ID. `Location ID` identifies a row in this file; a
re-survey of the same road gets new IDs and bins that no longer line up.

The target is one record per bin:

```
geometry           LineString, start to end of the bin
location_id        this file's ID for the bin
carriageway        divided A, divided B, undivided
star_rating        { car, car_raw }                  1 to 5
srs                { run_off..., head_on..., total } risk scores
roadworks          none | minor | major
speed_limit        a code, documented as km/h
roadside_*         { distance, object } for each side of the road
...                and every other coded column, as an enum
```